<a href="https://colab.research.google.com/github/BosunL/SEA-VQA/blob/main/Ateso_Gemma_3_4B_SEA_CVQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Visual Question Answering — Gemma 3 4B (Ateso)

---



Evaluating Gemma 3 4B on a custom Kenyan cultural dataset with 50 questions in Ateso across 14 images.

### Install Necessary Libraries

First, we need to install the `transformers` library to access pre-trained VQA models and `Pillow` for image processing.

In [ ]:
# Install transformers and other necessary libraries
!pip install -q transformers accelerate pillow matplotlib requests


### Import Libraries

Import the required modules from `transformers` and `PIL` (Pillow).

In [ ]:
#STEP 1: Install and Import Libraries

import random
import io
import requests
import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from huggingface_hub import login

print("Libraries imported successfully.")


### Authenticate with Hugging Face

Accept the Gemma license at https://huggingface.co/google/gemma-3-4b-it then run this cell.

In [ ]:
# ============================================================
# STEP 2: Authenticate with Hugging Face
# ============================================================
# You need to accept the Gemma license at:
# https://huggingface.co/google/gemma-3-4b-it
# Then paste your HF token below or use the Colab secrets manager

login(token="INSERT TOKEN")


### Load Pre-trained VQA Model and Processor

We'll use the `Google/gemma-3-4b-it` model, which is a powerful VQA model.

In [ ]:
# ============================================================
# STEP 3: Load Model and Processor
# ============================================================

print("Loading Gemma 3 4B model and processor...")

MODEL_ID = "google/gemma-3-4b-it"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto"
)

print(f"Model loaded on: {model.device}\n")


### Load Custom Dataset (Ateso)

All 50 Ateso questions from the Kenyan culture dataset. Images are fetched from Google Drive by file ID and cached to avoid redundant downloads.


**PSA: Gemma doesn't handle Google Sheets well, so we had to manually write out the questions. This is why we ran multiple sanity checks on the model, making there be more steps in the process.**

In [ ]:
# ============================================================
# STEP 4: Load Custom Kenyan Culture Dataset (Ateso)
# ============================================================
# Images are loaded from Google Drive using the file ID.
# A cache dict avoids re-downloading the same image multiple times.

CUSTOM_DATA = [
    {
        "ID": "custom_001",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "native_question": "Ibukito alu ajsii aputo, enayrit kesi nyena kakwap kalo? Inyena nesii asoma kesii?",
        "correct_native": "Euderu. Ejeo kaladoi kiton amusogon edumunun  kotogoi lukalong itosomao akipiet inyamen Itosomao adakite eariot kosokoni katoni kore",
        "wrong_native_o1": "Ibukit luka mesob. Anyart apee akou kanyaiti kape isutite, iranginino luchaete na itomayo akiria kadakite inyamene katikere",
        "wrong_native_o2": "Ibukito luka bolga. Ejeo kesi alumni kosere kaleoburo cuti. Ichasit kesi aso iboete cuti aoaki neuja.  Asoma kesi kogwelete kanamikwangere akilimia itogoi, akigwa iboro,  katoni atongitongoi",
        "wrong_native_o3": "Eguniati ngoli titosmomao akitowonio ichoko luepelel ingarakini akitene ichoko lukakiraa kotetenyi alumni iraito lukejoka",
    },
    {
        "ID": "custom_002",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "native_question": "Inyena asoma kejerikana nukobegete",
        "correct_native": "Akigwa akinyet kedia, akipii katoni inyamet luewokito",
        "wrong_native_o1": "Etapana adakari ibukito",
        "wrong_native_o2": "Itosomao akigwa akiria nu esudite",
        "wrong_native_o3": "Akituuja akijara kikole luepalala ka kiringi",
    },
    {
        "ID": "custom_003",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "native_question": "Inyena itosomaete itugna adakaia iboro kogwelete kaluchisi.",
        "correct_native": "Akarigo. Atapana cuti akitosom kanukamum akinyet . Edaki idwenyana idisi na elosete itunga lukegwelete katapan abongori ore.",
        "wrong_native_o1": "Atuktuk. Itapana  etachiti kene na  etapana adakia iboro luepu.",
        "wrong_native_o2": "Emotokaa. Kanueraitere nakapoloni itomate adakia inyamata luipwaka katoni luitijimete inyamene ayari namakegwelete",
        "wrong_native_o3": "Ebodaboda. Eraiti achakadu ikokorori kiyare kere  akarigo na elosi katipe noi. Itosomate kitunga kalidarito alosite. Edumuni cuti kochalo kaoni otauni.",
    },
    {
        "ID": "custom_004",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "native_question": "Nyena adumunet kegunieti?",
        "correct_native": "Eguniete etosomao akigwa emogo kekurididi. Itenio kaplastic katoni namaejaite kokinga baba",
        "wrong_native_o1": "Eguniati Etosomao akidaka emuchere loipoyo nginiparan kapolou",
        "wrong_native_o2": "Eguniati kachuti edakiti amaragwe nakaoni kakigwa tenan",
        "wrong_native_o3": "Eguniati ngoli titosmomao akitowonio ichoko luepelel ingarakini akitene ichoko lukakiraa kotetenyi alumni iraito lukejoka",
    },
    {
        "ID": "custom_005",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "native_question": "Abesin nejii kowai kamaa kateten erumit inyamat. Inyamt ngulu itente kinyena, eponee ali litenitere?",
        "correct_native": "Akimaa. Ipikino pwap  aadis adis kuju kakaratasit, kakinyekin  kowokoto kakolong, kosodi kobilibilite, idis dis.",
        "wrong_native_o1": "Emugati. Itenio kakiria  kipikinete akinyet numwaka, kere konyee kinyekinete kobu.",
        "wrong_native_o2": "Ichips. Etubio idisdis kakipikini toma  Okiala loka kim loko kuju chut,  paka nam edaunore akipee kijokis,  kakipikin akipii kotauchi.",
        "wrong_native_o3": "Emuchere luwowate kakinyet. Emuchere nuh mema kipoote  epyakion kwapu, kakichikakin kakitoi, kakipore",
    },
    {
        "ID": "custom_006",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "native_question": "Eponee likakiten inyamet a luipwaka kosokoni, ipyanikini iboro luipwaka lukakitijijim. Inyamata luh kijokis  itodikiti akapolok kajenaa kegwelet kalii?",
        "correct_native": "Itunga luka india luitenete ererwe  kiton lukebele luboyete okidee loka Africa",
        "wrong_native_o1": "Imusuguu  atodikini itung lukoo Kenya akiporee emugatii kakii wowa.",
        "wrong_native_o2": "Abunore kitunga kaluko Yoruba toma agwelet kokenya.",
        "wrong_native_o3": "Itunga luka amerika ayauni ekiponee kes lokaki gwaa  inyamat apak nepolo kakinyogokin akitosom",
    },
    {
        "ID": "custom_007",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "native_question": "Kiteyoo iboro nuejasii aputo ngolii, enyariti inyena?",
        "correct_native": "Agorogor",
        "wrong_native_o1": "Lokakitemio",
        "wrong_native_o2": "Alangir",
        "wrong_native_o3": "Iraito",
    },
    {
        "ID": "custom_008",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "native_question": "Anddot loka chumat katon iboro,  itingito iboro bala  erangi,  ichoko katoni akinyet kedia. Kipokona iboru ngulu itosomayo eponali. Nyena itodikit.",
        "correct_native": "Akipima kakigwaa. Itodikini akinyogoo  akitosom iboro.",
        "wrong_native_o1": "Akisiboi boro. Itodikit achwaa nakitete nakkisiboi iboro.",
        "wrong_native_o2": "Akitodiar akigiriasha kiboro. Itodikini  egwelet.",
        "wrong_native_o3": "Akipyakin inyamat kobeikino adakar. Atodikin  akidak inyamat.",
    },
    {
        "ID": "custom_009",
        "Category": "Image 3 Questions",
        "file_id": "1LagZEieb5SzUKwQ2jLfgGOXhjchC-_B0",
        "native_question": "Nyo ejaree ibore eniii akipi?",
        "correct_native": "Ibore ngini namaa enyemere ikaru idis kau ngina. Kipokona ibore nideuna erait ichumai agwoete akipii. Nuh ejasii anam loka Victoria, atupa kakiyatakin kakipii, elamarosii akipii amaat kitungaa.",
        "wrong_native_o1": "Adukio ibore eni kanu aporiat ikweny, amaat, kakiboi kooria kitunga, kakishal asoma nukakor.",
        "wrong_native_o2": "Iboree ngini, itosomao akiiny ikolee. Enetii kopwap ekamii inyamt lukakipii. Aronus konye ekitenio ken enyarautii ikweny.",
        "wrong_native_o3": "Ibore kachan isomae bala Iboro lukarukwor. Ikwenye erukuorete  iboro llukarioko lu ebunete amunakin lukainyok naminyoto.",
    },
    {
        "ID": "custom_010",
        "Category": "Image 3 Questions",
        "file_id": "1LagZEieb5SzUKwQ2jLfgGOXhjchC-_B0",
        "native_question": "Nyena ataker kikweny nitodikit ikweny nikironon kakou katon akwan  naka kwangan nejii kuju kiding kaputoo ngolii?",
        "correct_native": "Acuroit",
        "wrong_native_o1": "Amorocokin na aanywang",
        "wrong_native_o2": "Amorocokin na ejaatar keda eitoli lo eajany",
        "wrong_native_o3": "Ekeny lo lo ka akipi lo akirion",
    },
    {
        "ID": "custom_011",
        "Category": "Image 3 Questions",
        "file_id": "1LagZEieb5SzUKwQ2jLfgGOXhjchC-_B0",
        "native_question": "Nyena atakere kikweny atakanete kaputo kana?",
        "correct_native": "Acuroit, Amorocokin na aanywang, Amorocokin na ejaatar keda eitoli lo eajany, Enyilil",
        "wrong_native_o1": "Ewalu, Ekeny lo lo ebeleketi amun, Ekeny lo lo ka anam lo ka asirikiny, Amorocokin na edit",
        "wrong_native_o2": "Ekeny lo lo eduki akipi, Ekeny lo lo egwedi apany, Ekeny lo lo ka akipi lo ejaatar keda ituben, Aruut",
        "wrong_native_o3": "Abanga na aanywang elosikinit kanyen, Ekeny lo lo ka akipi lo ejaatar keda arem na arengan kanyen, Among'ot, Amorocokin na korit",
    },
    {
        "ID": "custom_012",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "native_question": "Ateter neetodunii aputoo nah , inyena chede nesikitsomate tunga",
        "correct_native": "Akiiny",
        "wrong_native_o1": "Aboyikini",
        "wrong_native_o2": "Akilosia Akaaree nakapolon",
        "wrong_native_o3": "Akitolosia Iboro",
    },
    {
        "ID": "custom_013",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "native_question": "Akipii nuh, luetodunii enyariti nyena.",
        "correct_native": "Anam loka Victoria",
        "wrong_native_o1": "Akare loka Hindi",
        "wrong_native_o2": "Anam napoleon lokared",
        "wrong_native_o3": "Anam nakapolon",
    },
    {
        "ID": "custom_014",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "native_question": "Aberu nigwoii, Kingaren kokedweny, enapit enaga lokagogong loka kenya. Enyariti enaga ngolii nyena?",
        "correct_native": "Kitenge",
        "wrong_native_o1": "Agbada",
        "wrong_native_o2": "Kente",
        "wrong_native_o3": "Amauti",
    },
    {
        "ID": "custom_015",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "native_question": "Kianyuyo agirit nejii ateter nejii aputoo ngolii, aputa nah keyaroo nama walii akwap loka Kenya?",
        "correct_native": "Dunga Beach",
        "wrong_native_o1": "Hippo Point",
        "wrong_native_o2": "Kit-Mikayi",
        "wrong_native_o3": "Impala Sanctuary",
    },
    {
        "ID": "custom_016",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "native_question": "Ikoni asoma ka mpessa ibelekini akijara kangotelei kokenya aii?",
        "correct_native": "Elonyaki eponee lokapeseii kokenya, Kanuu alachakii lumemmu kejasi kakaunt  abeikini  akigwaa, akiyakani,  kaki diyauni abolai alomuni kasiimu",
        "wrong_native_o1": "Edeuni aosma ken agelakina noi kanuu itsosomatete tunga liye ettiyakuna bon.",
        "wrong_native_o2": "Ayawu etanikina nedit, kanuu itete tunga luipwaka  kiteete adumunet kasoma ken",
        "wrong_native_o3": "Ebelok chut akiro kapesei kokenya kuju pwap, edipo apesei edachari pwap, nuh ebunete atupa kapesei akilom kalomar kogwelet.",
    },
    {
        "ID": "custom_017",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "native_question": "Nyena ekiror enyarite itogoi luchis luke bele nguluu kaduket?",
        "correct_native": "Kiosks",
        "wrong_native_o1": "Abutii",
        "wrong_native_o2": "Agwoete nakakirabaraba",
        "wrong_native_o3": "Eduka lochii.",
    },
    {
        "ID": "custom_018",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "native_question": "Nyena adumnute  ka cocacola ajaut kakigirashia Kotogo kangoli, katoni apwap nako Kenya kere.",
        "correct_native": "Ekampuni loka cocacal nesi le polo apwap  loka africa, ejasi kebele  lokapolon Kisumu. Ainakit ituga iboro luka long, bala elilinya, kakilony  echamuto itunga  egwelanar esoda kes, kakitodiar alama kes.",
        "wrong_native_o1": "Agwel cocala esokoni kisodai kijokis eteter erauni nesii loke bele kaniko kuju. Itodoarito  ebele kesi kotogoi kangulu teter itunga  egwelete iboro kes.",
        "wrong_native_o2": "Cocacola itopoloi ebele kesii koponee kaloka gwelakini itunga inagai, kakigir otogoi kes, kakitodiar iboro keso. Kakikono neni etopolo ebele loka kenya, itodikit ikampunio  epoloyete  atipet toma Okenya",
        "wrong_native_o3": "Emam adumunet kalama kangin, itung aloka keya elakarkina akipyakin alama ngini otogoi, amotokai kes kiton agwelet kes kalong",
    },
    {
        "ID": "custom_019",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "native_question": "Nyena ibore nebeikini akisomakini kariri ketogo nejii aputo na?",
        "correct_native": "Agwele  iboro idis kinyamen",
        "wrong_native_o1": "Agwelanar inagai lumojong",
        "wrong_native_o2": "Ainakin  emusolo katoni apaplai",
        "wrong_native_o3": "Akipikin epone lokalosia namiche",
    },
    {
        "ID": "custom_020",
        "Category": "Image 6 Questions",
        "file_id": "1nATb92NkD0KfWqsiOsPCfeo9s-DXZB-m",
        "native_question": "Nyena amoru nuitodute kaputo kana na nyena adumunte kes?",
        "correct_native": "Amoru nuka kalshiam nuko Kodiaga. Amoru nuh kakijara kanaka sem, anyemete aberu nepotiete  akiyatakini akwana kes ekalshiam, kanuka kingarakin akoyo kikoku katoken.",
        "wrong_native_o1": "Amoru nuka granait  nuitosomao aduko. Kanu alangiri egranait kibote de ding itosomao kesii aduko.",
        "wrong_native_o2": "Amoru  nuka laim, nue dummuno kotolima loka anam naka Victoria. Enonok naesi edipori itosomao kakoru kitoni akiduk. Itosomao akimada akipii kakitene ekeya kikyela.",
        "wrong_native_o3": "Amoru neelemute  kama ebokere emaram. Ekepite kanu emamm ketapana akitosom kesii kasodit. Duchu itosomao kanu emasiete akipii na adukio nuko pwap.",
    },
    {
        "ID": "custom_021",
        "Category": "Image 6 Questions",
        "file_id": "1nATb92NkD0KfWqsiOsPCfeo9s-DXZB-m",
        "native_question": "Nyena imoru luchusi lukaringak nwejasii okartasi, kosodi itenio kesii eponali?",
        "correct_native": "Emaide. Iwawo kakim nejasii kabalang, kowokoto.",
        "wrong_native_o1": "Amarage kepiro. Esekio kakitabaun kakipii.  Itukulauno kakipi idaikan itomon kanuka lemar adwarus. Kitolosi akiporee bala asaa epee kakim kadiosii kononokar",
        "wrong_native_o2": "Emaide. Epacho emaido. Akipii nujeasii amot nwemaka , etayatakino iboro lukajijim, kokulaete. Etolosit emadie akulau isaanii iuni, kedaunosii, utabauno isaan idisi, kolom abalang tenaa. Elemare akipii kijokis kagwelar.",
        "wrong_native_o3": "Amaide loka Macadamia. Epacho kaki pietar. Epeyoo kakim koaikan bala itomon karee. Apakii kanaa ipikino  abalang  kakipyakin ochwei, kagwelar.",
    },
    {
        "ID": "custom_022",
        "Category": "Image 6 Questions",
        "file_id": "1nATb92NkD0KfWqsiOsPCfeo9s-DXZB-m",
        "native_question": "Nyena ejii andoot naka kwangan?",
        "correct_native": "Imints loka KSL",
        "wrong_native_o1": "Amoru luka glass",
        "wrong_native_o2": "Amoru nuka Tanzania",
        "wrong_native_o3": "Amoru nuitente nuka gwelanar.",
    },
    {
        "ID": "custom_023",
        "Category": "Image 7 Questions",
        "file_id": "1jLX5cxvjljTk0PtMOaXFG5m055KSCptC",
        "native_question": "Nyena iboro enapite kaputo kanaa?",
        "correct_native": "Asikat naka amujaj loka Owalii",
        "wrong_native_o1": "Asikat naka Hula kahiko",
        "wrong_native_o2": "Asikat naka piupiu",
        "wrong_native_o3": "Liku",
    },
    {
        "ID": "custom_024",
        "Category": "Image 7 Questions",
        "file_id": "1jLX5cxvjljTk0PtMOaXFG5m055KSCptC",
        "native_question": "Nyena atakere eboikit anapa loka ibor kaluu?",
        "correct_native": "Dholuo",
        "wrong_native_o1": "Imaasai",
        "wrong_native_o2": "Isomalii",
        "wrong_native_o3": "Imeeru",
    },
    {
        "ID": "custom_025",
        "Category": "Image 7 Questions",
        "file_id": "1jLX5cxvjljTk0PtMOaXFG5m055KSCptC",
        "native_question": "Worii kapaki kanii enapere iboroo iluu",
        "correct_native": "Enapete aberu  ileete. Enapio apakii nakaki numnum emanyit, kakidowuni ikoku. Anunuk  kamujajua elachankini itwanu akilej.",
        "wrong_native_o1": "Itosomate idwee akilejia, aoaki nakaki numunumu akidounio, katoni apak nakakitubio kaluka tumunak.",
        "wrong_native_o2": "Enapete aberu  ki kiliyok  esaa lokakinumnum. Itosomao apakii nakakinumnum iboro lukedeke katoni akinumnum amojong. Emunt ateker amuja egwoikit akiro kedeke, na eraiti akinap.",
        "wrong_native_o3": "Enapete aberu  kanukaminakes bon. Itosomao akitoduni irangino. Akitene ne jokuna nah idiport enapite iboro luh akitodiar amina kitunga kalakara.",
    },
    {
        "ID": "custom_026",
        "Category": "Image 8 Questions",
        "file_id": "1ZuWu-TWfzj_jE7c0Wgs8W3936X1Du-nQ",
        "native_question": "Nyena ibor eruchokite kaunot kanakiryonon",
        "correct_native": "Echangwee Lokekitoi.",
        "wrong_native_o1": "Ekuridi lokekwangan",
        "wrong_native_o2": "Alaboro nukawoko.",
        "wrong_native_o3": "Echangwe loka anam nakapolon",
    },
    {
        "ID": "custom_027",
        "Category": "Image 8 Questions",
        "file_id": "1ZuWu-TWfzj_jE7c0Wgs8W3936X1Du-nQ",
        "native_question": "Nyena atenikina kiboro erucjokite kaunot kankiryonon",
        "correct_native": "Echangwe Lokakiswaga akwan. Itabuono kakipi, Eboste tenan kalipo, katoni  iboro luiswagere ibor luenyamere.",
        "wrong_native_o1": "Luinyamat. Einakini iboro luedikte akwan  bala Potassium katoni Vitamin B6. Akinyam iboro ngulu einakini akok nejokuna, kosodi isomae bala inyamen nuelae kakwan.",
        "wrong_native_o2": "Akipii kes nwe tapana adumum. Itingito akipii nejasii kiminerals, edwenyo  elemunete akipii ngulu. Ejasii kosodii nelemenere akipii kijokis kakwan.",
        "wrong_native_o3": "Akiduk. Echilaro kokiding akinyalakin kesimit akilelebia aduyoka otogoi katoni kuju ketogo.  Anonnok kes idiporit itosomao adukia itogoi lu iboyete",
    },
    {
        "ID": "custom_028",
        "Category": "Image 8 Questions",
        "file_id": "1ZuWu-TWfzj_jE7c0Wgs8W3936X1Du-nQ",
        "native_question": "Nyena iboro nu itogwoete nateten nyena asom kiboro kangulu?",
        "correct_native": "Anya nuka akorobos lu emameun apasao, lu eswamao nuka amunokin imanyas, ijokin, keda itatago. Ipedori bobo aenit keda airiit ecae nuka aodokin.",
        "wrong_native_o1": "Akwi nuka amuka na anyorire. Ejaatar keda aswamisinei ipu, naitisai aponia aitagulaun ecae lo amugit. Esubio bobo amunokin iasama keda aanyam na ititini aitogogong, keda aodokin arebeit (ingata) na ebeit aitit keda aiki iboro lu ititini.",
        "wrong_native_o2": "Mukeu, lo aponia eeswamao aoroi na agogong ka na itatene. Esubio bobo aodokin itatago keda itatago nuka kiondo.",
        "wrong_native_o3": "Itatago nuka apira nuka amukian aswam kotoma ateker. Konyout amunokin keda aitogogong keda amonikin emisago nuka amukian adukanat nepepe. Esubio bobo aitopol, acuran, keda aiki kwan.",
    },
    {
        "ID": "custom_029",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "native_question": "Ani kanyara na ka igwenya na itodunitai kotoma epicha?",
        "correct_native": "Omena keda emukene",
        "wrong_native_o1": "Ngege keda ewape",
        "wrong_native_o2": "Fulu keda amuny",
        "wrong_native_o3": "Ngege ka anam lo Magadi Oreochromis alcalicus",
    },
    {
        "ID": "custom_030",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "native_question": "Ani aswam na ka igwenya na akaulo nuka amukit aijany?",
        "correct_native": "Aanyam nuka itunga keda aanyam nuka itisai ka ore",
        "wrong_native_o1": "Aanyam aitogogong aitit igwenya nuka amina",
        "wrong_native_o2": "Aitingis kede aiita itingisin nuka amukian apasao",
        "wrong_native_o3": "Aitabul keda amugit ecae lo amukian adukanat",
    },
    {
        "ID": "custom_031",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "native_question": "Ani aswam na ka eipone lo lo ka aitoregeg boroke?",
        "correct_native": "Aenit keda aitukulun akula nuka aanyam, na itirini aitogogong aitit aswam na ka ekaulo ejok",
        "wrong_native_o1": "Aemwony ka aitorot kere imanyas keda agwelingot, aitiki itoroboto nuka aitun itingisin nuka amukian apasao",
        "wrong_native_o2": "Aitegulaun ecae lo amugit lo ka akwap aitogogong aitit anyam keda aiyalama",
        "wrong_native_o3": "Ainonok aitogogong aitit acurakin keda itugorisi lu amukian adukanat",
    },
    {
        "ID": "custom_032",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "native_question": "Ani aswam na ka aberu na itodunitai kotoma epicha?",
        "correct_native": "Aitutun igwenya lukaditot kotoma egwang lo katenait aitwan kokiolong amunokin",
        "wrong_native_o1": "Aitutun igwenya lukaditot kotoma egwang lo katenait areit acurakin keda aanyam lo ka abwong",
        "wrong_native_o2": "Aitutun igwenya lukaditot kotoma egwang lo katenait aanyun lu ejaatar keda itingisin keda lu emameun",
        "wrong_native_o3": "Aitelekun akwito keda amun nuka igwenya lu aponia itelekun aitit amukit aijany",
    },
    {
        "ID": "custom_033",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "native_question": "Ani eipone lo eeswamao itogoi lu?",
        "correct_native": "Idulit keda akito, emwonyit keda anyonit nuka aite keda aoo. Eswi bobo idulit keda amukian, iwakit keda anya na anyorire keda aoroi, ka emwonyit keda anyonit nuka aite aiki ekisina ka akipi lo akwap aitorot.",
        "wrong_native_o1": "Idulit keda akito, emwonyit keda simeeti keda amukian nuka akito. Eswi bobo idulit keda amukian, iwakit keda anya na awoleto keda emuse, ka emwonyit keda simeeti aitogogong.",
        "wrong_native_o2": "Asirikitin itogogongit keda simeeti na ka anyonit. Imukian igwedi keda ekisina lo ka akito nuka akimait aiki ikurut. Itwolo na keditot keda ikomian nuka aiki ekuwam lo alilim. Eswi bobo idulit keda amukian ka iwakit keda anya na anyorire keda aoroi.",
        "wrong_native_o3": "Idulit keda aswaki ka iwakit keda anyonit nuka akorobos aponia aitogogong. Atwolo na koki iwakit keda anya keda anyonit na ebeit aiywun ejok aibo ka aitwong. Anya na awoleto ebiit kidama ka eswi lo amukian lo emuse.",
    },
    {
        "ID": "custom_034",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "native_question": "Inai eduki itogoi lu kotoma ateker na ka Aluo?",
        "correct_native": "Imongin keda iaberu nuka ateker kere ejaatar keda aswamisinei aenit amorocokin",
        "wrong_native_o1": "Iaberu nuka ateker",
        "wrong_native_o2": "Okoki lo apol lo aberu na nasodit",
        "wrong_native_o3": "Imongin nuka ateker (okilen keda imongin lukaditot)",
    },
    {
        "ID": "custom_035",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "native_question": "Inai eboie kotogo lo eponi lo ka ekaulo apol?",
        "correct_native": "Aberu na nasodit",
        "wrong_native_o1": "Okilen",
        "wrong_native_o2": "Imongin lukaditot",
        "wrong_native_o3": "Ipejon lu aponia bwan",
    },
    {
        "ID": "custom_036",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "native_question": "Ani akwap na aponia itelekunai epicha lo?",
        "correct_native": "Diat Kit Mikayi",
        "wrong_native_o1": "Diat Maasai Mara",
        "wrong_native_o2": "Diat Mt. Kenya",
        "wrong_native_o3": "Diat anam lo ka akwap na a Kenya lo katenait kidama",
    },
    {
        "ID": "custom_037",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "native_question": "Ani amaduli na ka a Kenya na itodunitai kotoma epicha?",
        "correct_native": "Od Mikayi",
        "wrong_native_o1": "Manyatta / Enkaji",
        "wrong_native_o2": "Borana / Rendille Domes",
        "wrong_native_o3": "Itogoi nuka Swahili Coral Huts",
    },
    {
        "ID": "custom_038",
        "Category": "Image 11 Questions",
        "file_id": "15A5HdRDwqnfbXayubjIrqsj5HR6dW0Kb",
        "native_question": "Ani iasama lu eanywunitai kidama ka aoroi?",
        "correct_native": "Lu ka ateker kere ka iisuben lu kapuaka",
        "wrong_native_o1": "Lu ka imongin lukaditot bon",
        "wrong_native_o2": "Lu ka iaberu bon",
        "wrong_native_o3": "Lu ka imongin bon",
    },
    {
        "ID": "custom_039",
        "Category": "Image 11 Questions",
        "file_id": "15A5HdRDwqnfbXayubjIrqsj5HR6dW0Kb",
        "native_question": "Ani itodunit epicha lo kotoma aswam na ka itunga ka ore keda iasama?",
        "correct_native": "Elwatar keda akanin ka aenit kidama ka aoroi lo ka aitwan kokiolong",
        "wrong_native_o1": "Emait iasama keda iaswamak lu ka ailwat keda aitwan",
        "wrong_native_o2": "Erodit iasama kwan iaswama nuka akanin",
        "wrong_native_o3": "Eiit iasama ka amukit aijany amonikin emisago",
    },
    {
        "ID": "custom_040",
        "Category": "Image 11 Questions",
        "file_id": "15A5HdRDwqnfbXayubjIrqsj5HR6dW0Kb",
        "native_question": "Ani itisai lu eanywunitai koki ka aoroi lo ka iasama, ka ani aswamikebe?",
        "correct_native": "Irai abangat na ka Muscovy na eeswamao nuka itunga ka ore keda amonikin iasama. Irai ikwenta lu emameun iisuben lu eeswamao nuka aanyam, akwito, keda aiki ikurut lu aponia bwan.",
        "wrong_native_o1": "Irai ikoko lu eeswamao nuka aswam na ka akorobos. Emonikit akwito nuka aswam kotoma acurakin keda aanyam, ka amukit aijany nuka amonikin emisago.",
        "wrong_native_o2": "Irai abangat na ka Khaki Campbell na elosit katenait anam lo ka Victoria aitun ikurut kotoma anya nuka itunga ka ore.",
        "wrong_native_o3": "Irai abangat na ka areit na anyarait na aponia bwan na aanyam anya keda ikurut lu ka akwap.",
    },
    {
        "ID": "custom_041",
        "Category": "Image 12 Questions",
        "file_id": "1UlOl57GPNS9cr8MoyA_xCKfbCvNVRPve",
        "native_question": "Ani aswam na ka okilen lo itodunitai kotoma epicha?",
        "correct_native": "Eiramit apira kidama ka atatago na ka atuboi aduko",
        "wrong_native_o1": "Ainyasum atuboi na ka akito na abila",
        "wrong_native_o2": "Areit aiki atuboi kotoma akipi",
        "wrong_native_o3": "Aduko atatago na ka akito na awoleto",
    },
    {
        "ID": "custom_042",
        "Category": "Image 12 Questions",
        "file_id": "1UlOl57GPNS9cr8MoyA_xCKfbCvNVRPve",
        "native_question": "Aswam na, kotoma omarisin kere, elosit aitun...",
        "correct_native": "Iiapas lu keditot aitun iapas lu kapuaka",
        "wrong_native_o1": "Aparan iasama nepepe",
        "wrong_native_o2": "Aparanin lu keditot bon",
        "wrong_native_o3": "Ikarin lu keditot bon",
    },
    {
        "ID": "custom_043",
        "Category": "Image 12 Questions",
        "file_id": "1UlOl57GPNS9cr8MoyA_xCKfbCvNVRPve",
        "native_question": "Ani aswam na ka okilen lo?",
        "correct_native": "Ekaaswaman lo ka atuboin",
        "wrong_native_o1": "Einjinia lo ka akwap",
        "wrong_native_o2": "Ekaalwaran lo ka igwenya",
        "wrong_native_o3": "Ekaaswaman lo ka apira",
    },
    {
        "ID": "custom_044",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "native_question": "Epicha lo itodunit eipone lo lo ka aiki lo apol elosit katenait...",
        "correct_native": "Boda-boda, tuktuk, keda emuuga",
        "wrong_native_o1": "Pikipiki, matatu, keda egari lo ka apira lo agogong",
        "wrong_native_o2": "Ibasikeli keda pikipiki",
        "wrong_native_o3": "Emuuga, pikipiki, keda atuboi na apol na ka akipi",
    },
    {
        "ID": "custom_045",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "native_question": "Ani eboiet nuka pikipiki nuka itolosi itunga kotoma epicha lo?",
        "correct_native": "Boda-boda",
        "wrong_native_o1": "Autobikes",
        "wrong_native_o2": "Okada",
        "wrong_native_o3": "Cruisers",
    },
    {
        "ID": "custom_046",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "native_question": "Ani eipone lo ka pikipiki lo amina ka lo apol aswam kotoma aibo na?",
        "correct_native": "Honda CG125",
        "wrong_native_o1": "Ducati",
        "wrong_native_o2": "BMW Motorrad",
        "wrong_native_o3": "Honda NC750X",
    },
    {
        "ID": "custom_047",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "native_question": "Ekaaiton ka ekaatwan lo ka iasama katenait amukit aijany itoriti elosi aiki...",
        "correct_native": "Pikipiki keda ibasikeli",
        "wrong_native_o1": "Emuuga",
        "wrong_native_o2": "Emuuga lu apolok lu ka aiki iboro",
        "wrong_native_o3": "Tuktuk",
    },
    {
        "ID": "custom_048",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "native_question": "Epicha lo aponia itelekunai kotoma...",
        "correct_native": "Aibo na ka amait amauta nuka emuuga",
        "wrong_native_o1": "Aibo na ka aanyam",
        "wrong_native_o2": "Aibo na ka ainyasum emuuga",
        "wrong_native_o3": "Aibo na ka amait akipi kotoma icupa",
    },
    {
        "ID": "custom_049",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "native_question": "Ani ejaatatar kotoma ecupa kidama ka emesa lo ka aitwong?",
        "correct_native": "Aboslo na ka ailwat iboro kere",
        "wrong_native_o1": "Aboslo na ka ailwat iasama ka ore",
        "wrong_native_o2": "Soda lo amugit",
        "wrong_native_o3": "Akipi",
    },
    {
        "ID": "custom_050",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "native_question": "Ani aodokin na ka ekaaswaman lo ejaatar keda eoroi lo arengan?",
        "correct_native": "National Oil Corporation of Kenya",
        "wrong_native_o1": "Rubis Energy Kenya",
        "wrong_native_o2": "KFC",
        "wrong_native_o3": "Safaricom",
    },
    {
        "ID": "custom_051",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "native_question": "Ani aswam na ka itunganan lo?",
        "correct_native": "Amait amauta nuka emuuga keda amunokin emisago nuka amukit aijany",
        "wrong_native_o1": "Aiki aanyam katenait itunga lu eeswamao",
        "wrong_native_o2": "Aiki emuuga keda aswam na ka ainyasum iboro lu abila",
        "wrong_native_o3": "Ailwat akwap keda aswam na ka aiki akorobos kere",
    },
]

def load_drive_image(file_id, _cache={}):
    """Download a Google Drive image by file ID, with caching."""
    if file_id in _cache:
        return _cache[file_id]
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        img = Image.open(io.BytesIO(r.content)).convert("RGB")
        _cache[file_id] = img
        return img
    except Exception as e:
        print(f"  Warning: could not load image {file_id}: {e}")
        return Image.new("RGB", (224, 224), color="gray")

print(f"Custom dataset: {len(CUSTOM_DATA)} questions across 14 images.")


### Preview Dataset

In [ ]:
# ============================================================
# STEP 5: Preview Dataset — First 2 Samples
# ============================================================
print("--- Sample Preview: Custom Kenyan Dataset (Ateso) ---\n")

for i, sample in enumerate(CUSTOM_DATA[:2]):
    print(f"Sample {i+1}")
    print(f"  ID:         {sample['ID']}")
    print(f"  Category:   {sample['Category']}")
    print(f"  Question:   {sample['native_question']}")
    print(f"  Correct:    {sample['correct_native']}")
    print(f"  Wrong 1:    {sample['wrong_native_o1']}")
    print(f"  Wrong 2:    {sample['wrong_native_o2']}")
    print(f"  Wrong 3:    {sample['wrong_native_o3']}")
    print()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for i, sample in enumerate(CUSTOM_DATA[:2]):
    img = load_drive_image(sample['file_id'])
    axes[i].imshow(img)
    axes[i].set_title(f"Sample {i+1}\n{sample['Category'][:20]}", fontsize=8)
    axes[i].axis("off")
plt.suptitle("Custom Kenyan Dataset (Ateso) — First 2 Samples", fontsize=11)
plt.tight_layout()
plt.show()


### Helper Functions

In [ ]:
# ==========================================================
# STEP 6: Define Helper Functions
# ===========================================================

def build_mcqa_prompt(question, choices):
    """
    Format question + choices into a multiple choice prompt.
    Instructs the model to respond with only A, B, C, or D.
    """
    prompt = (
        "You are answering a visual multiple choice question. "
        "You must respond with ONLY a single letter: A, B, C, or D.\n\n"
        "Example:\n"
        "Question: What color is the sky?\n"
        "A. Red\nB. Blue\nC. Green\nD. Yellow\n"
        "Answer: B\n\n"
        "Now answer this question:\n"
        f"Question: {question}\n"
    )
    for label, choice in zip(["A", "B", "C", "D"], choices):
        prompt += f"{label}. {choice}\n"
    prompt += "Answer:"
    return prompt


def extract_predicted_label(predicted_text, choices):
    """
    Extract predicted label (A/B/C/D) from model output.
    Strategy 1: look for A/B/C/D letter in model output.
    Strategy 2: if no letter found, match answer text directly.
    """
    for label in ["A", "B", "C", "D"]:
        if label in predicted_text.upper():
            return label
    for label, choice in zip(["A", "B", "C", "D"], choices):
        if choice.lower() in predicted_text.lower():
            return label
    return None


### Gemma Inference Function

In [ ]:
# ==========================================================
# STEP 7: Helper Function to Run the Gemma Model
# ===========================================================

def run_gemma(image, prompt, processor, model):
    """
    Run Gemma 3 4B inference on an image + text prompt.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": prompt}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device, dtype=torch.bfloat16)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    input_len = inputs["input_ids"].shape[-1]
    predicted_text = processor.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    ).strip()

    return predicted_text


### Evaluation Function

In [ ]:
# ============================================================
# STEP 8: Define Evaluation Function
# ============================================================

def evaluate_custom_dataset(data, processor, model):
    correct = 0
    predictions = []
    total = len(data)

    print(f"\nRunning evaluation on {total} samples...")
    print("-" * 40)

    for i, sample in enumerate(data):
        if (i + 1) % 10 == 0 or i == 0:
            print(f"  Processing sample {i+1}/{total}...")

        question    = sample["native_question"]
        correct_ans = sample["correct_native"]
        choices     = [
            correct_ans,
            sample["wrong_native_o1"],
            sample["wrong_native_o2"],
            sample["wrong_native_o3"]
        ]
        random.shuffle(choices)
        correct_label = ["A", "B", "C", "D"][choices.index(correct_ans)]

        prompt = build_mcqa_prompt(question, choices)

        try:
            image           = load_drive_image(sample["file_id"])
            predicted_text  = run_gemma(image, prompt, processor, model)
            predicted_label = extract_predicted_label(predicted_text, choices)
            is_correct      = (predicted_label == correct_label)
            if is_correct:
                correct += 1
        except Exception as e:
            print(f"  Error on sample {i+1}: {e}")
            predicted_text  = "ERROR"
            predicted_label = None
            is_correct      = False

        predictions.append({
            "sample_id":       sample["ID"],
            "category":        sample["Category"],
            "question":        question,
            "choices":         {l: c for l, c in zip(["A","B","C","D"], choices)},
            "correct_answer":  correct_ans,
            "correct_label":   correct_label,
            "predicted_text":  predicted_text,
            "predicted_label": predicted_label,
            "is_correct":      is_correct
        })

    accuracy = (correct / total) * 100 if total > 0 else 0
    return accuracy, predictions


### Sanity Check (5 Samples)

In [ ]:
# ============================================================
# STEP 9: Sanity Check on First 5 Samples
# ============================================================
print("\nRunning sanity check on 5 samples...")

_, sanity_predictions = evaluate_custom_dataset(
    CUSTOM_DATA[:5], processor, model
)

print("\nSanity check results:")
for i, pred in enumerate(sanity_predictions, 1):
    status = "✓" if pred["is_correct"] else "✗"
    print(f"  [{status}] Q{i}: {pred['question']}")
    print(f"        Predicted: {pred['predicted_label']} "
          f"| Correct: {pred['correct_label']} "
          f"| Model output: {pred['predicted_text']}")

print("\nIf results look reasonable, proceed to full evaluation...\n")


### Full Evaluation (50 Questions)

In [ ]:
# ============================================================
# STEP 10: Run Full Evaluation (50 questions)
# ============================================================
accuracy, all_predictions = evaluate_custom_dataset(
    CUSTOM_DATA, processor, model
)


### Results

In [ ]:
# ============================================================
# STEP 11: Print Final Results
# ============================================================
print("\n" + "="*60)
print(f"  Final Results — Gemma 3 4B | Kenyan Culture Dataset (Ateso)")
print("="*60)

category_scores = {}
for pred in all_predictions:
    cat = pred["category"]
    if cat not in category_scores:
        category_scores[cat] = {"correct": 0, "total": 0}
    category_scores[cat]["total"] += 1
    if pred["is_correct"]:
        category_scores[cat]["correct"] += 1

print("\nPer-image accuracy:")
for cat, scores in sorted(category_scores.items()):
    cat_acc = (scores["correct"] / scores["total"]) * 100
    print(f"  {cat:<45} {cat_acc:.1f}%  ({scores['correct']}/{scores['total']})")

print(f"\nOverall Accuracy:        {accuracy:.2f}%")
print(f"Correct:                 {sum(p['is_correct'] for p in all_predictions)}/{len(all_predictions)}")
print(f"Random chance baseline:  25.00% (4 choices)")
print("="*60)


### Visualise Sample Results

In [ ]:
# ============================================================
# STEP 12: Visualise Results
# ============================================================

import math

# Create a dictionary to group predictions by image file_id
image_performance_map = {}
for i, sample in enumerate(CUSTOM_DATA):
    file_id = sample["file_id"]
    if file_id not in image_performance_map:
        image_performance_map[file_id] = {"image": None, "predictions": []}
    # Load image only once per file_id
    if image_performance_map[file_id]["image"] is None:
        image_performance_map[file_id]["image"] = load_drive_image(file_id)

    # Associate prediction with the correct image based on sample ID
    # Note: all_predictions is 1:1 with CUSTOM_DATA, so we can use index 'i'
    image_performance_map[file_id]["predictions"].append(all_predictions[i])

# Get a sorted list of unique file_ids for consistent plotting order
unique_file_ids = sorted(image_performance_map.keys())
num_unique_images = len(unique_file_ids)

# Determine grid dimensions (e.g., 4 columns)
ncols = 4
nrows = math.ceil(num_unique_images / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
# Flatten the axes array for easier iteration if it's a multi-row grid
axes = axes.flatten() if nrows > 1 or ncols > 1 else [axes] # Handle single subplot case

for i, file_id in enumerate(unique_file_ids):
    img_data = image_performance_map[file_id]
    img = img_data["image"]
    predictions_for_image = img_data["predictions"]

    correct_count = sum(p['is_correct'] for p in predictions_for_image)
    total_questions = len(predictions_for_image)
    image_accuracy = (correct_count / total_questions) * 100 if total_questions > 0 else 0

    # Use the category of the first question associated with this image as a general image label
    image_category = predictions_for_image[0]["category"]

    axes[i].imshow(img)
    color = "green" if image_accuracy == 100 else ("red" if image_accuracy == 0 else "orange")
    axes[i].set_title(
        f"{image_category}\nAcc: {image_accuracy:.1f}% ({correct_count}/{total_questions})",
        fontsize=8, color=color
    )
    axes[i].axis("off")

# Hide any unused subplots
for j in range(num_unique_images, len(axes)):
    if j < len(axes): # Ensure index is valid
        fig.delaxes(axes[j])

plt.suptitle(
    f"Gemma 3 4B — Kenyan Culture Dataset (Ateso) | Overall Accuracy: {accuracy:.1f}% | Random baseline: 25%",
    fontsize=12
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to make space for suptitle
plt.show()